# 训练

发起一次训练 + 检查这一次跑得对不对。**判读结果去 [`MSN_compare_runs.ipynb`](MSN_compare_runs.ipynb)。**

1 控制面板（**只改这里**）· 2 预检 · 3 训练 · 4 自检 · 5 存档 · 附录 A 数据准备 · 附录 B 相对原 demo 的改动

> ⚠️ ① kernel 选 `comp0190-msn`　② **本 notebook 不要建模型**（训练子进程要 15.5/24 GiB，占了就 OOM）　③ 想断连不中断就用终端，第 1 节会打印完整命令

## 1. 控制面板 ⬅️ 每次只改这里

| 想干什么 | 怎么填 |
|---|---|
| **重复一次已有 run** | `FROM_RUN="tie_qk"`、`RUN_NAME="tie_qk_r2"`、`EXTRA_FLAGS=[]` |
| 试新配置 | `FROM_RUN=""`，自己写 `EXTRA_FLAGS` |
| 在已有配置上改一项 | `FROM_RUN="cd_rep05_full"` + `EXTRA_FLAGS=["--dcd-lambda","2"]`（会警告：不再是严格重复） |

`FROM_RUN` 照抄那个 run 记录的 18 个超参。**重复实验必须用它** —— 手抄 flag 正是重复实验悄悄变成"另一个实验"的地方。

⚠️ `RUN_NAME` 撞名会被拒绝启动（`cd_only` 的权重就被同名重跑覆盖过）。

In [1]:
import os, sys, json, subprocess, shutil
import numpy as np

# ============================ 每次改这里 ============================
RUN_NAME    = "notext_r2"      # 这一轮的名字，会成为 experiments/msn_skullfix/<name>/
FROM_RUN    = "notext"         # 照抄这个 run 的全部超参；留空 "" 则用下面的 flags
EXTRA_FLAGS = []               # 例：["--loss", "cd", "--repulsion-weight", "0.5"]
# ===================================================================

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
PY_MSN = "/root/miniconda3/envs/comp0190-msn/bin/python"
OUT_DIR = os.path.join(REPO, "experiments", "msn_skullfix", RUN_NAME)

CMD = [PY_MSN, "src/models/train_skullfix.py", "--run-name", RUN_NAME]
if FROM_RUN:
    CMD += ["--from-run", FROM_RUN]
CMD += EXTRA_FLAGS

print("REPO      ", REPO)
print("输出目录   ", os.path.relpath(OUT_DIR, REPO))
print("\n完整命令（要在终端跑就复制这一行）:")
print("  " + " ".join(CMD))
print("\n可用的 flag: python src/models/train_skullfix.py --help")

REPO       /root/comp0190-organ-completion
输出目录    experiments/msn_skullfix/notext_r2

完整命令（要在终端跑就复制这一行）:
  /root/miniconda3/envs/comp0190-msn/bin/python src/models/train_skullfix.py --run-name notext_r2 --from-run notext

可用的 flag: python src/models/train_skullfix.py --help


## 2. 预检

数据 / 配对对齐 / 显存 / run 名，四项都只用 CPU。

配对对齐看的是「每个 GT 点到最近输入点」：中位数应≈采样间距，只有尾部（缺损区）该大。**中位数大 = 两朵点云不在同一坐标系**，那是数据 bug，训出来的一切都不可信。

In [2]:
CACHE = os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz")
assert os.path.exists(CACHE), f"缓存不存在，先跑附录 A：\n{CACHE}"

data = np.load(CACHE)
ids, inputs, gt = data["ids"], data["inputs"], data["gt"]
scale_mm = float(data["scale_mm"].mean())
print(f"数据   {len(ids)} 对 | input {inputs.shape} | gt {gt.shape} | scale {scale_mm:.1f} mm")


def _nn_dist(query, ref, chunk=1024):
    """逐块纯 numpy 最近邻距离。这个 kernel 故意不装 scipy（版本被 numpy<2 +
    tensorflow<2.16 钉死），一个自检不值得动它。"""
    ref2 = (ref ** 2).sum(1)
    out = np.empty(len(query), dtype=np.float64)
    for i in range(0, len(query), chunk):
        q = query[i:i + chunk]
        d2 = (q ** 2).sum(1)[:, None] - 2.0 * (q @ ref.T) + ref2[None, :]
        out[i:i + chunk] = np.sqrt(np.maximum(d2.min(1), 0.0))
    return out


nn = _nn_dist(gt[0].astype(np.float64), inputs[0].astype(np.float64)) * scale_mm
ok = np.median(nn) < 4.0
print(f"配对对齐 skull_{ids[0]}: 中位 {np.median(nn):.2f} mm（共享表面，应 < 4）"
      f" | p99 {np.percentile(nn, 99):.2f} mm（缺损区，应 > 10）  {'✅' if ok else '❌ 数据有问题'}")

print("\n显存:")
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
      " ← used 应接近 0；不为 0 说明有别的 kernel 占着，训练会 OOM")

taken = [f for f in ("run.json", "best.h5", "history.csv")
         if os.path.exists(os.path.join(OUT_DIR, f))]
print(f"\nrun 名 '{RUN_NAME}': " + ("✅ 可用" if not taken else
      f"❌ 已被占用（{', '.join(taken)}）—— 换个名字，训练脚本会拒绝启动"))

数据   100 对 | input (100, 4096, 3) | gt (100, 6144, 3) | scale 103.8 mm
配对对齐 skull_000: 中位 2.75 mm（共享表面，应 < 4） | p99 20.14 mm（缺损区，应 > 10）  ✅

显存:
1 MiB, 24564 MiB  ← used 应接近 0；不为 0 说明有别的 kernel 占着，训练会 OOM

run 名 'notext_r2': ✅ 可用


## 3. 训练

约 9 s/epoch，近期各轮 222~411 轮，即 **35~60 分钟**。

| 停止机制 | 值 | 说明 |
|---|---|---|
| `EarlyStopping` | patience 20 | 真正的停止信号，并恢复最优权重 |
| `ReduceLROnPlateau` | patience 10（推导） | **必须小于早停 patience，否则永不触发** |
| `--epochs` / `--minutes` | 500 / 150 min | 兜底。⚠️ **撞 `--epochs` 上限的 run 作废** |

已有各轮（`run.json` 口径，比 compare notebook 的逐颅骨口径系统性低约 0.09mm，**别混引**）：

| run | 配置 | epochs | CD_t |
|---|---|---:|---:|
| `cd_rep05_full` | **CD+rep** | 256 | **6.267** |
| `cd_rep05_r2` | 同上，重复 | 249 | 6.274 |
| `lr_fix_only` | CD+DCD | 279 | 6.317 |
| `rep_w05` | CD+DCD+rep | 222 | 6.326 |
| `cd_only` | CD 单独 | 305 | 6.353 |
| `notext` | 去文本分支 | 380 | 6.225 |
| `tie_qk` / `tie_qk_r2` | Q/K 绑定 ❌ 已否决 | 411 / 246 | 6.198 / 6.319 |
| `pp_attn` | 逐点注意力 ❌ 已否决 | 355 | 6.581 |

⚠️ 轮数差别很大而报告的是"全程最优"，跨 run 比较要看 compare notebook 的**同轮次表**。

In [3]:
proc = subprocess.Popen(CMD, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print("\nreturn code:", proc.returncode)

[--from-run] 复制自 experiments_log/notext/run.json
[--from-run] 复制了 18 个字段: batch_size=4, config='paper', dcd_lambda=1.0, dcd_weight=1.0, early_stop_patience=20, epochs=500, fold=0, loss='cd', lr=0.0003, minutes=150.0, n_folds=0, no_text=True, per_point_attn=False, repulsion_k=4, repulsion_r0=2.0, repulsion_weight=0.5, seed=42, tie_qk_init=False
[--from-run] 记录里没有这些字段，按「它当时生效的默认值」补: per_point_attn, tie_qk_init
2026-08-24 18:55:42.970708: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-24 18:55:42.970738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-24 18:55:42.971517: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plug

## 4. 本轮自检

问的是"这次跑得对不对"，不是"结果好不好"。四个红灯：

| 检查 | 正常 | 不正常说明什么 |
|---|---|---|
| 停止原因 | `EarlyStopping` | 撞 `--epochs` 上限 → **这轮作废**，调高上限重跑 |
| LR 衰减 | 6~9 次 | 0 次 = 衰减没触发，"最优值"是随机游走上的最小值，不可读 |
| 末 30 轮 std | < 0.02 mm | 大（~0.25mm）= 没退火好，同上 |
| val/train | 1.0~1.2× | 本模型历来 1.03~1.19×，从没过拟合过 |

In [4]:
import pandas as pd

meta = json.load(open(os.path.join(OUT_DIR, "run.json")))
hist = pd.read_csv(os.path.join(OUT_DIR, "history.csv"))
best_ep = int(hist["val_loss"].idxmin()) + 1
n_ep = len(hist)
lr_drops = sum(1 for i in range(1, n_ep)
               if hist["lr"][i] < hist["lr"][i - 1] - 1e-12)
late_std = (hist["val_cd_t_metric"] * meta["scale_mm"]).tail(30).std()
ratio = meta["final"]["val_cd_t_metric"] / meta["final"]["cd_t_metric"]

# 先判早停：它是决定性的（最后 patience 轮没有改善）。反过来先判上限会误伤 ——
# 早期的 run.json 没记 --epochs，回退值可能比它实际用的上限小（cd_only 就被误报过）。
if n_ep - best_ep == meta["early_stop_patience"]:
    stop = "✅ EarlyStopping"
elif "epochs" not in meta:
    stop = "⚠️ run.json 没记 --epochs 上限，无法判定是否被截断"
elif n_ep >= meta["epochs"]:
    stop = "❌ 撞 --epochs 上限被截断 —— 这一轮不可引用，调高上限重跑"
else:
    stop = "⚠️ 被 --minutes 墙钟预算掐停"

print(f"run          {RUN_NAME}   ({meta['loss']}, rep={meta['repulsion_weight']:g}, "
      f"tie_qk={meta.get('tie_qk_init', False)}, use_text={meta.get('use_text', True)})")
print(f"停止原因      {stop}")
print(f"轮数          {n_ep}（最优在第 {best_ep} 轮，之后 {n_ep - best_ep} 轮无改善）")
print(f"LR 衰减       {lr_drops} 次      {'✅' if lr_drops >= 5 else '❌ 太少，检查配置'}")
print(f"末 30 轮 std  {late_std:.4f} mm  {'✅ 已退火' if late_std < 0.02 else '❌ 没退火好'}")
print(f"val/train     {ratio:.2f}×       {'✅ 正常' if ratio < 1.25 else '⚠️ 偏高'}")
print(f"\nbest val CD_t {meta['best_val_cd_t_mm']:.3f} mm   ← run.json 口径，"
      f"和 compare notebook 的逐颅骨口径差约 0.09mm，不要混引")

run          notext_r2   (cd, rep=0.5, tie_qk=False, use_text=False)
停止原因      ✅ EarlyStopping
轮数          227（最优在第 207 轮，之后 20 轮无改善）
LR 衰减       8 次      ✅
末 30 轮 std  0.0055 mm  ✅ 已退火
val/train     1.18×       ✅ 正常

best val CD_t 6.248 mm   ← run.json 口径，和 compare notebook 的逐颅骨口径差约 0.09mm，不要混引


## 5. 存档

`run.json` + `history.csv` 进 git —— 权重丢了能重训，实验记录丢了就复现不出对比表。

跑完还要：`experiments_log/README.md` 加一行 · `devlog.md` 追加一节 · `git commit`。

In [5]:
dst = os.path.join(REPO, "experiments_log", RUN_NAME)
os.makedirs(dst, exist_ok=True)
for f in ("run.json", "history.csv"):
    shutil.copy2(os.path.join(OUT_DIR, f), os.path.join(dst, f))
print(f"已存档 -> experiments_log/{RUN_NAME}/  ({', '.join(os.listdir(dst))})")
print("\n别忘了：experiments_log/README.md 加一行 + devlog.md 追加一节 + git commit")

已存档 -> experiments_log/notext_r2/  (run.json, history.csv)

别忘了：experiments_log/README.md 加一行 + devlog.md 追加一节 + git commit


## 下一步

判读 → [`MSN_compare_runs.ipynb`](MSN_compare_runs.ipynb)　|　表面质量 / mesh → [`MSN_surface_quality.ipynb`](MSN_surface_quality.ipynb)

---

## 附录 A — 数据准备

**只在改点数 / 样本量时跑**，当前缓存已是 100 对。纯 CPU，不碰显存。
`WORKERS=8` 是实测最优（这活是内存带宽密集，12 比 8 慢、24 比 4 还慢）。

In [ ]:
N_SAMPLES, N_DENSE, N_IN, N_OUT, WORKERS = 0, 16384, 4096, 6144, 8

_out = os.path.join(REPO, "data", "cache", f"skullfix_pairs_{N_IN}_{N_OUT}.npz")
_p = subprocess.Popen(
    [PY_MSN, "src/data/prepare_skullfix.py", "--n-samples", str(N_SAMPLES),
     "--n-dense", str(N_DENSE), "--n-in", str(N_IN), "--n-out", str(N_OUT),
     "--workers", str(WORKERS), "--out", _out],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in _p.stdout:
        print(line, end="")
finally:
    _p.wait()
print("\nreturn code:", _p.returncode)

## 附录 B — 本实现相对原 demo 的改动

细节见 `src/models/msn_skullfix.py` 的模块 docstring。

| | 改动 | 为什么 |
|---|---|---|
| **A** | 配对对齐：从 defective 推出**唯一一个**相似变换，同时作用于两朵云 | 各自独立归一化会让配对的两朵云落到**不同坐标系**（实测质心偏 3.6%，GT→input 最近距离被抬高 32%） |
| **B** | 距离矩阵改用 `\|a\|²−2a·b+\|b\|²` | demo 的 tile 法在 batch 8 / 6144 点下单个张量就 12GB；改完 187M 模型在单卡 4090 上 372ms/步、15.5GiB，**训得动了** |
| **C** | 默认损失 `cd_dcd` 而非纯 DCD | DCD 有界 [0,2]，随机初始化时两个因子同时消失，loss 钉在 1.9995 训不动 |
| **D** | 推理改用固定种子的 stateless 采样 | demo 的有状态抽样让同一输入两次调用差 1.03，指标不可复现 |
| **E** | 其它 | lr 1e-7→3e-4 + warmup · batch 8→4（显存）· 按 id 显式划分（Keras 是先切尾部再打乱，会泄漏）· BERT 预计算 · checkpoint 用旧式 `.h5`（新格式连 Adam 动量一起存，2.25GB）· 体素间距按 nrrd 头做 matmul |